In [ ]:
from pathlib import Path

import io
import onnx
import torch
# from onnxsim import simplify

from hailo_sdk_client import ClientRunner

import sys
sys.path.append('..')
from model import Model

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Generating forward-only graph

In [ ]:
# Create a classifier instance
device = "cpu"
batch_size = 64
num_classes = 37
channels = 3
img_h, img_w = 128, 128
frozen_layer_num = 10

model_name = 'resnet18' # resnet18, resnet34, mobilenetv2_100, efficientnet_b2, visformer_tiny

# artifacts path
artifacts_path = Path(f'../artifacts/{model_name}_frozen_layer_{frozen_layer_num}_classes_{num_classes}/hailo')
artifacts_path.mkdir(parents=True, exist_ok=True)

pt_model = Model(model_name=model_name, num_classes=num_classes, channels=channels, frozen_layer_num=frozen_layer_num, img_size=img_h)
print(f'Model trainable parameters: {count_parameters(pt_model)}')

for param in pt_model.named_parameters():
    if 'frozen_features' in param[0] or 'bn' in param[0] or 'downsample.1' in param[0]:
        param[1].requires_grad = False
    else:
        param[1].requires_grad = True

torch.save(pt_model, artifacts_path / f'{model_name}.pth')
pt_model = torch.load(artifacts_path / f'{model_name}.pth')
for param in pt_model.named_parameters():
    if param[1].requires_grad:
        print(param[0].ljust(40), '->\t', param[1].requires_grad)

print(f'Model trainable parameters: {count_parameters(pt_model)}')

In [ ]:
# Generate a random input.
model_inputs = (torch.randn(batch_size, channels, img_h, img_w, device='cpu'),)

model_outputs = pt_model(*model_inputs)
if isinstance(model_outputs, torch.Tensor):
    model_outputs = [model_outputs]
    
input_names = ["input"]
output_names = ["output"]
dynamic_axes = {"input": {0: "batch_size"}, "output": {0: "batch_size"}}

f = io.BytesIO()
torch.onnx.export(
    pt_model,
    model_inputs,
    f,
    input_names=input_names,
    output_names=output_names,
    opset_version=17,
    do_constant_folding=False,
    # training=torch.onnx.TrainingMode.PRESERVE,
    dynamic_axes=dynamic_axes,
    export_params=True,
    # keep_initializers_as_inputs=False,
)

onnx_model = onnx.load_model_from_string(f.getvalue())
# print(f'Simplifying ONNX {model_name} model...')
# eval_model, check = simplify(onnx_model)
onnx.save(onnx_model, artifacts_path / f'{model_name}.onnx')

#### Parsing ONNX to Hailo

In [ ]:
chosen_hw_arch = 'hailo8'
runner = ClientRunner(hw_arch=chosen_hw_arch)

model_path = artifacts_path / f'{model_name}.onnx'
start_node_names = ['input']
end_node_names = ['/frozen_features/frozen_features.10/Add']
net_input_shapes = {'input': [1, channels, img_h, img_w],}

In [ ]:
hn, npz = runner.translate_onnx_model(
    str(model_path),
    model_name,
    start_node_names=start_node_names,
    end_node_names=end_node_names,
    net_input_shapes=net_input_shapes,
)

In [ ]:
hailo_har_model_path = artifacts_path / f'{model_name}.har'
runner.save_har(hailo_har_model_path)

In [ ]:
# !hailo profiler {hailo_har_model_path}

#### Model optimization

In [ ]:
import numpy as np
from tqdm import tqdm
from PIL import Image


images_list = list(Path('../data/ILSVRC2012_img_val/').rglob('*.JPEG'))

img_mean = [123.675, 116.28, 103.53]
img_std = [58.395, 57.12, 57.375]

# calib_dataset = np.random.randint(0, 255, (1024, img_h, img_w, channels))
# calib_dataset = np.random.random((1024, img_h, img_w, channels))

# calib_dataset = np.zeros((len(images_list), img_h, img_w, channels))
calib_set_size = 1024
calib_dataset = np.zeros((calib_set_size, img_h, img_w, channels))
for idx, img_path in enumerate(tqdm(sorted(images_list[:calib_set_size]))):
    img = np.array(Image.open(img_path).convert('RGB').resize((img_h, img_w)))
    img = (img.astype(np.float32) - img_mean) / img_std
    # img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0).astype(np.float32)
    calib_dataset[idx, :, :, :] = img

calib_set_path = Path('./calib_set.npy')
np.save(calib_set_path, calib_dataset)

calib_dataset = np.load(calib_set_path)
print(f'Calibration dataset shape: {calib_dataset.shape}')
print(f'Calibration dataset size: {calib_dataset.nbytes / 1024**2:.2f} MB')

In [ ]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)

# Batch size is 8 by default
# alls = 'normalization1 = normalization([123.675, 116.28, 103.53], [58.395, 57.12, 57.375])\n'

# Load the model script to ClientRunner so it will be considered on optimization
# runner.load_model_script(alls)

# Call Optimize to perform the optimization process
runner = ClientRunner(hw_arch=chosen_hw_arch, har=str(hailo_har_model_path))
runner.optimize(calib_dataset)

# Save the result state to a Quantized HAR file
quantized_har_model_path = str(hailo_har_model_path).replace('.har', '_quantized_model.har')
runner.save_har(quantized_har_model_path)

In [ ]:
# runner.analyze_noise(calib_dataset, data_count=16, batch_size=2)

In [ ]:
# !hailo profiler {quantized_har_model_path}

#### Quantized model compilation to HEF format

In [ ]:
runner = ClientRunner(har=quantized_har_model_path)
hef = runner.compile()

hef_model_path = quantized_har_model_path.replace('har', 'hef')
with open(hef_model_path, 'wb') as f:
    f.write(hef)

#### Export to ONNX with HailoOp operator

In [ ]:
onnx_model = runner.get_hailo_runtime_model()
onnx_artifacts_path = hef_model_path.replace('.hef', '_hailo.onnx')
onnx_file = onnx.save(onnx_model, onnx_artifacts_path)
print(f'ONNX with HailoOp saved as: {onnx_artifacts_path}')

## Generating training graph

Generating artifacts based on forward-only graph and the specified loss function using onnxblock library

In [ ]:
import onnx
import onnxruntime.training.onnxblock as onnxblock

In [ ]:
# Creating a class with a Loss function
class CustomTrainingBlock(onnxblock.TrainingBlock):
    def __init__(self):
        super(CustomTrainingBlock, self).__init__()
        self.loss = onnxblock.loss.CrossEntropyLoss()

    def build(self, output_name):
        return self.loss(output_name), output_name

In [ ]:
onnx_model = onnx.load_model(onnx_artifacts_path)

for param in onnx_model.graph.initializer:
    print(param.name)

In [ ]:
training_block = CustomTrainingBlock()
for param in onnx_model.graph.initializer:
    if 'frozen_features' in param.name or 'bn' in param.name or 'downsample.1' in param.name:
        training_block.requires_grad(param.name, False)
    else:
        print(param.name.ljust(40), '\t--->\tlearnable')
        training_block.requires_grad(param.name, True)

# Building training graph and eval graph
model_params = None
with onnxblock.base(onnx_model):
    build_output = training_block(*[output.name for output in onnx_model.graph.output])
    print('Graph output nodes:', build_output)
    training_model, eval_model = training_block.to_model_proto()
    model_params = training_block.parameters()

# Building the optimizer graph
optimizer_block = onnxblock.optim.AdamW()
with onnxblock.empty_base() as accessor:
    _ = optimizer_block(model_params)
    optimizer_model = optimizer_block.to_model_proto()

In [ ]:
# Saving all the files to use them later for the training.
onnxblock.save_checkpoint(training_block.parameters(), artifacts_path / 'checkpoint')

ir_version = 9

training_model.ir_version = ir_version
# print('Simplifying ONNX training model...')
# training_model, check = simplify(training_model)
# assert check, "Simplified ONNX training model could not be validated"
onnx.save(training_model, artifacts_path / 'training_model.onnx')

optimizer_model.ir_version = ir_version
onnx.save(optimizer_model, artifacts_path / 'optimizer_model.onnx')

eval_model.ir_version = ir_version
# print('Simplifying ONNX eval model...')
# eval_model, check = simplify(eval_model)
# assert check, "Simplified ONNX eval model could not be validated"
onnx.save(eval_model, artifacts_path / 'eval_model.onnx')

print(f'Artifacts saved in {artifacts_path} directory')